# 经典 TP 基线与 TorchTitan TP+SP

## 为什么需要分析 TP 通信

FSDP 解决了参数存放问题——但它不改变一个事实：每张卡仍然要独立完成一层完整的 GEMM 计算。当模型宽度增大（更大的 hidden size 或 intermediate size），单层 GEMM 本身就可能超出单卡的计算能力或显存带宽。

TP（Tensor Parallelism）的解决思路是**切分权重矩阵**：把一层的权重按列或按行切成 N 份，多个 rank 共同完成同一次矩阵乘法，各自只保存和计算一部分权重。但切分权重意味着每个 rank 的输出不再是完整的——有些切分方式产生的是局部结果（需要 all-reduce 求和），有些产生的是分片结果（可以直接传给下游）。

如果不理解哪种切分需要通信、通信发生在 forward 还是 backward、以及为什么 TP 的通信常常直接落在关键路径上，就无法判断什么时候该用 TP、什么时候该用 FSDP 替代。本节固定 Qwen3-1.7B 的 MLP 维度（H=2048, I=6144），只构造代表性的激活张量，不执行实际 MLP 计算。

> TP 的通信由张量布局转换决定。本节先用 **Partial → all-reduce → Replicate** 建立经典 TP 基线，再单独阅读当前 TorchTitan 默认启用 sequence parallel 时的 AG/RS 布局转换。

![TP 列切分与行切分](images/tp_collectives_zh.svg)

**图 2：** 上投影的列切分输出已经是合法的分片布局；下投影的部分输出必须通过 all-reduce 汇合。

## 路线

- **§1** — TP 的原理是什么？你会得到：列切分 vs 行切分、Partial vs Shard vs Replicate 三种布局、forward 和 backward 各自在哪个位置触发通信。
- **§2** — 每个 rank 手里有什么数据？你会得到：33.554 MB 的激活张量、列切分输出不通信、行切分输出必须 all-reduce。
- **§3** — 在两张 NPU 上测经典 all-reduce 基线，并从逐 rank 原始样本自动生成 median/P95。
- **§4** — 当前 TorchTitan TP+SP 怎么描述布局？你会看到 `Shard(sequence) → Replicate` 与 `Partial → Shard(sequence)`，以及为何实际账本必须由 resolved config/trace 确认。
- **§5** — TP 通信为什么容易暴露在关键路径上？你会看到：all-reduce 的结果依赖、异步调用的有限 overlap 窗口，以及通信账本如何核对调用次数。


## 1. 先建立经典 Megatron-style TP 基线

TP 切分的是**同一次矩阵乘法**：多个 rank 共同处理同一批 token，但各自只保存和计算一部分权重。通信由局部结果的布局决定。

- **列切分**：沿输出维切权重，各 rank 的输出是不同坐标上的 Shard，可直接交给逐元素激活和后续行切分层。
- **行切分**：沿输入维切权重，各 rank 得到同一输出坐标上的 Partial。经典基线用 all-reduce 做 SUM，并在所有 rank 上得到 Replicate。
- **反向**：经典布局下，列切分层的输入梯度贡献需要求和；行切分层的输入梯度保持分片。

因此，在不启用 sequence parallel、并把每个 row-parallel 输出恢复为 Replicate 的经典 Transformer 账本里，attention 与 MLP 各贡献一次 forward all-reduce 和一次 backward all-reduce，即每层 forward 2 次、backward 2 次。

**这四次 all-reduce 只是本节 microbenchmark 的经典基线，不是当前 TorchTitan 默认 TP+SP 的实际通信账本。** 当前 `ParallelismConfig.enable_sequence_parallel=True`；Qwen3 plan 让 norm 和部分 activation 保持 sequence-sharded，并在模块边界做 `Shard(sequence) ↔ Replicate`、`Partial → Shard(sequence)` 的布局转换，通常对应 all-gather/reduce-scatter。确切次数、shape 和是否融合必须以 resolved config 与 trace 为准。

In [ ]:
B, S, H, I = 2, 4096, 2048, 6144
tp, dtype_bytes = 2, 2

def mb(x):
    return x / 1_000_000

x_bytes = B * S * H * dtype_bytes
column_output_per_rank = B * S * (I // tp) * dtype_bytes
row_partial_per_rank = B * S * H * dtype_bytes

print(f'输入 X：{mb(x_bytes):.3f} MB')
print(f'列切分输出（每 rank）：{mb(column_output_per_rank):.3f} MB（不立即通信）')
print(f'行切分 partial output（每 rank）：{mb(row_partial_per_rank):.3f} MB')
print(f'两卡 all-reduce 逻辑张量：输入 {mb(row_partial_per_rank):.3f} MB，输出 {mb(row_partial_per_rank):.3f} MB')
print(f'两卡 all-reduce 每 rank 理想对端交换量：{mb(row_partial_per_rank):.3f} MB')

## 2. 在两张 NPU 上运行 all-reduce

真正需要通信的位置是行切分下投影的输出。直接运行下面的 Bash 单元。

In [ ]:
%%bash
set -euo pipefail

# 需要调度器已分配并暴露两张 NPU。
mkdir -p results
torchrun --standalone --nproc_per_node=2 scripts/tp_collectives.py \
  --output-json results/tp_classic_latest.json \
  --profile-dir results/profiles/tp_classic_latest

## 3. 经典 all-reduce 基线的结果口径

脚本保存全部 rank/iteration 到 `results/tp_classic_latest.json`，并直接打印：

- 每个 rank 的 median、P95 和 min–max；
- 对齐每个 iteration 后的慢 rank median/P95；
- rank median 差值；
- 按单向发送 payload / 慢 rank median 计算的 demo 有效 GB/s。

`results/profiles/tp_classic_latest/` 是另采的一次 HCCL trace，不参与 wall-time 统计。尚未运行时，本章不展示历史手抄数字。

这个 microbenchmark 只回答一个 `[2,4096,2048]` all-reduce 有多慢。把它乘以经典基线的每层 4 次，只能得到**零 overlap 串行估算**，不是训练 step time，更不是当前 TorchTitan TP+SP 的实测账本。

Async TP 通过分块 AG→GEMM 或 GEMM→RS 创造 overlap 窗口，但会切碎 GEMM 并引入资源争用。本教程没有与当前硬件、shape、world size 和软件版本匹配的端到端原始数据，因此不声明具体百分比收益。

## 4. 当前 TorchTitan Qwen3：TP 默认与 sequence parallel 配合

当前 `ParallelismConfig.enable_sequence_parallel` 默认是 `True`。Qwen3 的上游 `apply_non_moe_tp` 先令 embedding 输出和 norm 输入保持 `Shard(sequence)`，再在 attention/MLP 边界声明所需布局：

- `PrepareModuleInput`：把 `Shard(sequence)` 输入转换成投影所需的 `Replicate`，通常需要 all-gather；
- Q/K/V 与 MLP 上投影：`ColwiseParallel` 切输出维；
- O 投影与 MLP down 投影：`RowwiseParallel(output_layouts=Shard(sequence))`，把 Partial 结果转换为 sequence shard，通常需要 reduce-scatter；
- norm：`SequenceParallel`，继续在 sequence-sharded activation 上计算。

因此“每层 forward 2 次、backward 2 次 all-reduce”只能作为 `enable_sequence_parallel=False` 的经典基线。当前 TP+SP 的 AG/RS 次数、shape、反向逆变换和可能的融合，应从实际 resolved config 与双 rank trace 核对；不能用本节 all-reduce demo 代替。NPU Qwen3 wrapper 最终调用上游 parallel plan，CP 启用时再注入 NPU Ulysses 实现。

### 计算思考

如果 TP 并行度从 2 改为 4，而输入 `[2,4096,2048]` 保持不变，下面哪一项会变化：all-reduce 的张量形状、逻辑输入/输出字节数，还是只有每个 rank 的理想交换量？请根据数据归属规则回答，而不是背公式。

## 5. 经典 TP 基线的依赖与暴露时间

下面的账本只计算经典 Replicate/Partial 布局下一个 Qwen3 block 及 28 层主干的 all-reduce；它不是当前默认 TP+SP 的调用账本，通信总量也不能直接换算成训练 step time。

经典 TP 的 row-parallel 输出是 **Partial**，若下游要求 Replicate，就必须先拿到 all-reduce 结果。`async_op=True` 只能与不依赖该结果的本地计算并行；若下游立刻读取，最终仍会在 `wait` 处等待。sequence parallel 把一部分 `Partial → Replicate` 改为 `Partial → Shard(sequence)`，从而改变 collective 类型和 activation 所有权，但不会自动消除布局转换依赖。


### all-reduce 的 API 调用

经典 demo 使用 `dist.all_reduce(partial, op=dist.ReduceOp.SUM)`：每个 rank 提供同形状 tensor，SUM 后每个 rank 得到相同结果。异步版本返回 Work handle，只能与不依赖结果的工作重叠。TorchTitan 的实际 TP+SP 使用 DTensor placement/functional collective 保留布局与 autograd 语义，不应从这个朴素 API 反推实际 collective 列表。

## 经典 TP 中不同层的通信

在经典 Replicate/Partial 布局中，Q/K/V 投影通常列切分，O 投影通常行切分；MLP 的上、下投影同理。于是 row-parallel 的 partial output 做 forward all-reduce，column-parallel 的输入梯度贡献在 backward 求和。启用 sequence parallel 后，这些模块边界的目标布局发生变化，不能继续照搬四次 all-reduce。

TP degree 从 2 改为 4 时，经典 row-parallel all-reduce 的逻辑 tensor 仍可保持 `[2,4096,2048]`；变化的是权重/中间维度分片、算法步骤和每 rank wire payload。

普通 `dist.all_reduce` 是原地 side effect，不能简单当作自动可微的 DTensor 算子。TorchTitan 的 DTensor placement/functional collective 携带布局和 autograd 语义。阅读 trace 时，应区分朴素 all-reduce、AG/RS、可能的融合和框架层 layout redistribution。


### 28 层经典 TP 通信账本（公式验证）

下面只计算 `enable_sequence_parallel=False`、每层 forward/backward 各 2 次 all-reduce 的教学基线。它不代表当前 TorchTitan 默认 TP+SP；embedding、norm、vocab/loss 和所有实际布局转换应从 resolved config 与 trace 另行加入。

In [ ]:
LAYERS = 28
ALL_REDUCE_FWD_PER_LAYER = 2
ALL_REDUCE_BWD_PER_LAYER = 2

def gb(x):
    return x / 1_000_000_000

# ring all-reduce：每 rank 单向发送 2*(N-1)/N 个完整逻辑张量。
all_reduce_send_per_rank = 2 * (tp - 1) / tp * x_bytes
forward_send = LAYERS * ALL_REDUCE_FWD_PER_LAYER * all_reduce_send_per_rank
backward_send = LAYERS * ALL_REDUCE_BWD_PER_LAYER * all_reduce_send_per_rank
train_send = forward_send + backward_send

# 若 full activation checkpoint 重放每个 block 的完整 forward，
# 会再执行一遍 forward TP collectives；实际次数仍应由 trace 验证。
train_send_with_full_forward_recompute = train_send + forward_send

print(f'一次 all-reduce：每 rank 发送 {mb(all_reduce_send_per_rank):.3f} MB，接收量相同')
print(f'28 层 forward：{2 * LAYERS} 次 all-reduce，发送 {gb(forward_send):.3f} GB/rank')
print(f'28 层 backward：{2 * LAYERS} 次 all-reduce，发送 {gb(backward_send):.3f} GB/rank')
print(f'一次 forward+backward：{4 * LAYERS} 次 all-reduce')
print(f'  单向发送：{gb(train_send):.3f} GB/rank')
print(f'  单向接收：{gb(train_send):.3f} GB/rank')
print(f'  发送+接收：{gb(2 * train_send):.3f} GB/rank')
print(f'若完整重算 forward：单向发送约 {gb(train_send_with_full_forward_recompute):.3f} GB/rank')

代码账本用于验证经典 TP 的公式和每 rank payload。若已有 `results/tp_classic_latest.json`，可把其中的慢 rank median 乘以 4 得到单 block 的零 overlap 串行估算；没有本机 JSON 时不填历史数字。

当前 TorchTitan TP+SP 的性能结论必须来自对应配置的完整训练 trace 和 profiler-off 吞吐。经典 all-reduce 基线只能解释 Partial→Replicate 的成本，不能决定是否应启用 TP。


## 练习：选择与判断

1. （判断题）经典 Megatron-style TP 的 all-reduce 账本可以不经 resolved config 或 trace，直接当作当前 TorchTitan TP+SP 的实际通信账本。

2. （单选题）当前 TorchTitan TP 与 sequence parallel 配合时，常见的 layout redistribution 是什么？
    A. Shard(sequence) 与 Replicate/Partial 之间通过 all-gather 或 reduce-scatter 转换
    B. 所有层固定只做 broadcast
    C. 每层只做参数 all-gather
    D. 完全没有 collective

3. （判断题）TP degree 从 2 改为 4 时，逻辑 all-reduce 张量形状可以不变，但每个 rank 的理想交换比例会变化。

4. （判断题）Async TP 在任何 workload 和设备上都固定提高 8%，因此无需给出软件、shape 与 trace。

In [ ]:
!cat ./answer/07.03_answer.txt
